In [0]:
# Bloco 1 — Carregar Silver
df_silver = spark.table("workspace.default.silver_hr_attrition")
df_silver.display()


age,attrition,business_travel,daily_rate,department,distance_from_home,education,education_field,employee_count,employee_number,environment_satisfaction,gender,hourly_rate,job_involvement,job_level,job_role,job_satisfaction,marital_status,monthly_income,monthly_rate,num_companies_worked,over18,over_time,percent_salary_hike,performance_rating,relationship_satisfaction,standard_hours,stock_option_level,total_working_years,training_times_last_year,work_life_balance,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager
41,true,travel_rarely,1102,sales,1,2,life sciences,1,1,2,female,94,3,2,Sales Executive,4,Single,5993.0,19479,8,Y,true,11.0,3,1,80,0,8,0,1,6,4,0,5
49,false,travel_frequently,279,research & development,8,1,life sciences,1,2,3,male,61,2,2,Research Scientist,2,Married,5130.0,24907,1,Y,false,23.0,4,4,80,1,10,3,3,10,7,1,7
37,true,travel_rarely,1373,research & development,2,2,other,1,4,4,male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,Y,true,15.0,3,2,80,0,7,3,3,0,0,0,0
33,false,travel_frequently,1392,research & development,3,4,life sciences,1,5,4,female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,Y,true,11.0,3,3,80,0,8,3,3,8,7,3,0
27,false,travel_rarely,591,research & development,2,1,medical,1,7,1,male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,Y,false,12.0,3,4,80,1,6,3,3,2,2,2,2
32,false,travel_frequently,1005,research & development,2,2,life sciences,1,8,4,male,79,3,1,Laboratory Technician,4,Single,3068.0,11864,0,Y,false,13.0,3,3,80,0,8,2,2,7,7,3,6
59,false,travel_rarely,1324,research & development,3,3,medical,1,10,3,female,81,4,1,Laboratory Technician,1,Married,2670.0,9964,4,Y,true,20.0,4,1,80,3,12,3,2,1,0,0,0
30,false,travel_rarely,1358,research & development,24,1,life sciences,1,11,4,male,67,3,1,Laboratory Technician,3,Divorced,2693.0,13335,1,Y,false,22.0,4,2,80,1,1,2,3,1,0,0,0
38,false,travel_frequently,216,research & development,23,3,life sciences,1,12,4,male,44,2,3,Manufacturing Director,3,Single,9526.0,8787,0,Y,false,21.0,4,2,80,0,10,2,3,9,7,1,8
36,false,travel_rarely,1299,research & development,27,3,medical,1,13,3,male,94,3,2,Healthcare Representative,3,Married,5237.0,16577,6,Y,false,13.0,3,2,80,2,17,3,2,7,7,7,7


In [0]:
# Bloco 2 — Criar gold_attrition_metrics

from pyspark.sql import functions as F

df_gold_metrics = (
    df_silver
        .agg(
            F.count("*").alias("total_employees"),
            F.sum(F.col("attrition").cast("int")).alias("total_attrition"),
            (F.sum(F.col("attrition").cast("int")) / F.count("*")).alias("attrition_rate"),
            F.avg("monthly_income").alias("avg_monthly_income"),
            F.avg("age").alias("avg_age"),
            F.avg("years_at_company").alias("avg_years_at_company"),
            F.avg(F.col("over_time").cast("int")).alias("percent_overtime"),
            F.avg("job_satisfaction").alias("avg_job_satisfaction"),
            F.avg("work_life_balance").alias("avg_work_life_balance")
        )
)

df_gold_metrics.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_attrition_metrics")


In [0]:
# Bloco 3 — Criar gold_attrition_by_department

df_gold_dept = (
    df_silver
        .groupBy("department")
        .agg(
            F.count("*").alias("total_employees"),
            F.sum(F.col("attrition").cast("int")).alias("total_attrition"),
            (F.sum(F.col("attrition").cast("int")) / F.count("*")).alias("attrition_rate"),
            F.avg("monthly_income").alias("avg_monthly_income"),
            F.avg("age").alias("avg_age"),
            F.avg("job_satisfaction").alias("avg_job_satisfaction"),
            F.avg("years_at_company").alias("avg_years_at_company")
        )
)

df_gold_dept.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_attrition_by_department")


In [0]:
# Bloco 4 — Criar gold_attrition_by_job_role

df_gold_role = (
    df_silver
        .groupBy("job_role")
        .agg(
            F.count("*").alias("total_employees"),
            F.sum(F.col("attrition").cast("int")).alias("total_attrition"),
            (F.sum(F.col("attrition").cast("int")) / F.count("*")).alias("attrition_rate"),
            F.avg("monthly_income").alias("avg_monthly_income"),
            F.avg("years_in_current_role").alias("avg_years_in_role"),
            F.avg("job_satisfaction").alias("avg_job_satisfaction")
        )
)

df_gold_role.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_attrition_by_job_role")


In [0]:
# Bloco 5 — Validar as tabelas Gold

print("gold_attrition_metrics")
spark.table("workspace.default.gold_attrition_metrics").display()

print("gold_attrition_by_department")
spark.table("workspace.default.gold_attrition_by_department").display()

print("gold_attrition_by_job_role")
spark.table("workspace.default.gold_attrition_by_job_role").display()


gold_attrition_metrics


total_employees,total_attrition,attrition_rate,avg_monthly_income,avg_age,avg_years_at_company,percent_overtime,avg_job_satisfaction,avg_work_life_balance
1470,237,0.16122448979591836,6502.931292517007,36.923809523809524,7.0081632653061225,0.2829931972789116,2.7285714285714286,2.7612244897959184


gold_attrition_by_department


department,total_employees,total_attrition,attrition_rate,avg_monthly_income,avg_age,avg_job_satisfaction,avg_years_at_company
research & development,961,133,0.1383975026014568,6281.252861602497,37.0426638917794,2.7263267429760667,6.8647242455775235
sales,446,92,0.2062780269058296,6959.17264573991,36.54260089686099,2.7511210762331837,7.2847533632287
human resources,63,12,0.19047619047619047,6654.507936507936,37.80952380952381,2.6031746031746033,7.238095238095238


gold_attrition_by_job_role


job_role,total_employees,total_attrition,attrition_rate,avg_monthly_income,avg_years_in_role,avg_job_satisfaction
Research Director,80,2,0.025,16033.55,6.2875,2.7
Healthcare Representative,131,9,0.06870229007633588,7528.763358778626,4.862595419847328,2.786259541984733
Human Resources,52,12,0.23076923076923078,4235.75,3.1346153846153846,2.5576923076923075
Sales Executive,326,57,0.17484662576687116,6924.2791411042945,4.855828220858895,2.754601226993865
Manufacturing Director,145,10,0.06896551724137931,7295.137931034483,4.972413793103448,2.682758620689655
Research Scientist,292,47,0.16095890410958905,3239.972602739726,3.2739726027397262,2.7739726027397262
Sales Representative,83,33,0.39759036144578314,2626.0,2.0120481927710845,2.7349397590361444
Laboratory Technician,259,62,0.23938223938223938,3237.169884169884,3.200772200772201,2.6911196911196913
Manager,102,5,0.049019607843137254,17181.676470588234,6.450980392156863,2.7058823529411766


In [0]:
# Bloco 6 — Criar dim_department

from pyspark.sql import functions as F

df_dim_department = (
    df_silver
        .select("department")
        .distinct()
        .withColumn("department_id", F.monotonically_increasing_id())
)

df_dim_department.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_department")

spark.table("workspace.default.dim_department").display()


department,department_id
research & development,0
sales,1
human resources,2


In [0]:
# Bloco 7 — Criar dim_job_role

df_dim_job_role = (
    df_silver
        .select("job_role")
        .distinct()
        .withColumn("job_role_id", F.monotonically_increasing_id())
)

df_dim_job_role.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_job_role")

spark.table("workspace.default.dim_job_role").display()


job_role,job_role_id
Research Director,0
Healthcare Representative,1
Human Resources,2
Sales Executive,3
Manufacturing Director,4
Research Scientist,5
Sales Representative,6
Laboratory Technician,7
Manager,8


In [0]:
# Bloco 8 — Criar fact_attrition

# Carregar dimensões
dim_dept = spark.table("workspace.default.dim_department")
dim_role = spark.table("workspace.default.dim_job_role")

# Criar tabela fato
df_fact_attrition = (
    df_silver
        .join(dim_dept, "department", "left")
        .join(dim_role, "job_role", "left")
        .select(
            "employee_number",   # identificador do funcionário no dataset original
            "attrition",
            "age",
            "monthly_income",
            "years_at_company",
            "job_satisfaction",
            "work_life_balance",
            "over_time",
            "department_id",
            "job_role_id"
        )
)

df_fact_attrition.write.format("delta").mode("overwrite").saveAsTable("workspace.default.fact_attrition")

spark.table("workspace.default.fact_attrition").display()


employee_number,attrition,age,monthly_income,years_at_company,job_satisfaction,work_life_balance,over_time,department_id,job_role_id
1,true,41,5993.0,6,4,1,true,1,3
2,false,49,5130.0,10,2,3,false,0,5
4,true,37,2090.0,0,3,3,true,0,7
5,false,33,2909.0,8,3,3,true,0,5
7,false,27,3468.0,2,2,3,false,0,7
8,false,32,3068.0,7,4,2,false,0,7
10,false,59,2670.0,1,1,2,true,0,7
11,false,30,2693.0,1,3,3,false,0,7
12,false,38,9526.0,9,3,3,false,0,4
13,false,36,5237.0,7,3,2,false,0,1


In [0]:
# Bloco 10 — Validar Star Schema

print("dim_department")
spark.table("workspace.default.dim_department").display()

print("dim_job_role")
spark.table("workspace.default.dim_job_role").display()

print("fact_attrition")
spark.table("workspace.default.fact_attrition").display()

print("Verificar se todos os registros da fato possuem departamento e cargo válidos:")
spark.sql("""
SELECT 
    COUNT(*) AS total_registros,
    SUM(CASE WHEN department_id IS NULL THEN 1 ELSE 0 END) AS dept_null,
    SUM(CASE WHEN job_role_id IS NULL THEN 1 ELSE 0 END) AS role_null
FROM workspace.default.fact_attrition
""").display()


dim_department


department,department_id
research & development,0
sales,1
human resources,2


dim_job_role


job_role,job_role_id
Research Director,0
Healthcare Representative,1
Human Resources,2
Sales Executive,3
Manufacturing Director,4
Research Scientist,5
Sales Representative,6
Laboratory Technician,7
Manager,8


fact_attrition


employee_number,attrition,age,monthly_income,years_at_company,job_satisfaction,work_life_balance,over_time,department_id,job_role_id
1,true,41,5993.0,6,4,1,true,1,3
2,false,49,5130.0,10,2,3,false,0,5
4,true,37,2090.0,0,3,3,true,0,7
5,false,33,2909.0,8,3,3,true,0,5
7,false,27,3468.0,2,2,3,false,0,7
8,false,32,3068.0,7,4,2,false,0,7
10,false,59,2670.0,1,1,2,true,0,7
11,false,30,2693.0,1,3,3,false,0,7
12,false,38,9526.0,9,3,3,false,0,4
13,false,36,5237.0,7,3,2,false,0,1


Verificar se todos os registros da fato possuem departamento e cargo válidos:


total_registros,dept_null,role_null
1470,0,0
